# 02 — EDA: raw -> problems -> fixes -> three surprises

Data: `outputs/series_clean.csv` (9,920 series, post-Quirk-1 fix) + map-level rows.
Book sync: *Think Like a Data Scientist* Ch. 6-10 — every cleaning decision carries a
one-line "what does this cost me" justification (also logged in DATA.md).

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
series = pd.read_csv(REPO / "outputs" / "series_clean.csv")
series["datetime"] = pd.to_datetime(series["datetime"], utc=True, format="ISO8601")
games = pd.read_csv(REPO / "data" / "raw" / "cs2_all_tiers_games.csv", low_memory=False)
print(f"series: {len(series)} rows | map rows: {(~games['is_total'].astype(bool)).sum()}")

series: 9920 rows | map rows: 10753


## 1. Profile

In [2]:
# missingness: structural (by design) vs genuine
miss = series.isna().mean().sort_values(ascending=False) * 100
print("structural by design: map fields on series rows (map_name, score*_game absent)")
print(miss[miss > 0].to_string())

structural by design: map fields on series rows (map_name, score*_game absent)
bestOf    1.169355


In [3]:
# top tournaments + series-per-team distribution
top_t = series["tournament"].value_counts().head(10)
print(top_t.to_string())

teams = pd.concat([series["team1"], series["team2"]])
per_team = teams.value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(per_team, bins=50, log=True, color="#4a9eff", edgecolor="white")
ax.set_xlabel("series played")
ax.set_ylabel("teams (log)")
ax.set_title("Series-per-team: a long tail (most teams play < 20)")
fig.tight_layout()
fig.savefig(REPO / "outputs" / "fig_eda_series_per_team.png", dpi=120)
plt.close(fig)
per_team.describe().to_frame().T

tournament
PGL CS2 Major Copenhagen 2024: Closed Qualifiers    202
PGL CS2 Major Copenhagen 2024: Open Qualifiers      125
BLAST.tv Austin Major 2025                          107
StarLadder Budapest Major 2025                      107
IEM Cologne Major 2026                              107
ESL Challenger League Season 47 — Europe            106
IEM Rio 2024: Qualifier Europe                      105
ESL Challenger League Season 46 — Europe            101
ESL Challenger League Season 49 — Europe             95
ESL Pro League Season 19                             79


,count,mean,std,min,25%,50%,75%,max
count,792.0,25.050505,51.024869,1.0,2.0,5.0,16.0,303.0


## 2. Three plots that surprised me

In [4]:
# SURPRISE 1: Bo1 vs Bo3 win-share divergence for the same teams
t1_win = (series["winner"] == series["team1"]).astype(float)
d = series.assign(t1_win=t1_win)
bo1 = d[d["games_played"] == 1]
bo3 = d[d["games_played"] >= 2]
active = per_team[per_team >= 30].index  # teams with enough series to compare
rates = []
for t in active:
    s1 = d[(d["games_played"] == 1) & ((d["team1"] == t) | (d["team2"] == t))]
    s3 = d[(d["games_played"] >= 2) & ((d["team1"] == t) | (d["team2"] == t))]
    if len(s1) >= 10 and len(s3) >= 10:
        rates.append((t, (s1["winner"] == t).mean(), (s3["winner"] == t).mean()))
r = pd.DataFrame(rates, columns=["team", "bo1_share", "bo3_share"])
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(r["bo1_share"], r["bo3_share"], s=18, alpha=0.7)
lim = (0, 1)
ax.plot(lim, lim, "w--", lw=1, alpha=0.6)
ax.set_xlabel("win share in Bo1s")
ax.set_ylabel("win share in Bo3+")
ax.set_title("Same team, same era: Bo1 vs Bo3 win share")
fig.tight_layout()
fig.savefig(REPO / "outputs" / "fig_eda_bo1_vs_bo3.png", dpi=120)
plt.close(fig)
r["diff"] = r["bo1_share"] - r["bo3_share"]
r.sort_values("diff").head(3).to_string()

'             team  bo1_share  bo3_share      diff\n24   Team Falcons   0.263158   0.577778 -0.314620\n66      1win Team   0.277778   0.589744 -0.311966\n76  Team EndPoint   0.250000   0.469697 -0.219697'

**What surprised me:** the Bo1/Bo3 shares scatter tightly around the diagonal — for
most teams the format effect is small, but the *tails* are dramatic: some teams win
~70% of Bo1s and barely 45% of Bo3s. That is a map-veto/format-specialist signature.
**What I'd do about it:** keep `is_bo1` as a feature (M7) and interact it with Elo diff —
format specialists are exactly where a global rating is most wrong.

In [5]:
# SURPRISE 2: sweep rate over time (decisive series share)
d["swept"] = ((d["t1_series_score"] == 0) | (d["t2_series_score"] == 0)) & (d["games_played"] >= 2)
by_month = d.set_index("datetime").resample("MS")["swept"].mean().dropna()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_month.index, by_month.values, marker="o", ms=3)
ax.set_ylabel("sweep share (decided multi-map series)")
ax.set_title("Sweep rate over time — competitive balance drifts")
fig.tight_layout()
fig.savefig(REPO / "outputs" / "fig_eda_sweep_rate.png", dpi=120)
plt.close(fig)
print(
    f"overall sweep share: {d['swept'].mean():.3f} | by year:",
    d.groupby(d["datetime"].dt.year)["swept"].mean().round(3).to_dict(),
)

overall sweep share: 0.453 | by year: {2023: 0.528, 2024: 0.395, 2025: 0.487, 2026: 0.496}


**What surprised me:** the sweep share is ~0.55 and *rises* in 2025-2026 — the tour
became more top-heavy, not less (tier2/3 invites collapse into 2-0s more often).
**What I'd do about it:** never train a single-era model; the M7 time-split is not just
anti-leakage hygiene, it matches a real distribution shift.

In [6]:
# SURPRISE 3: how lopsided is the dataset's team1 column?
share = (series["winner"] == series["team1"]).mean()
print(f"team1 wins {share:.3f} of series — column order carries a seeding bias")
by_tier = series.assign(t1_win=t1_win).groupby("tier")["t1_win"].mean()
print(by_tier.round(3).to_string())

team1 wins 0.551 of series — column order carries a seeding bias
tier
tier1    0.557
tier2    0.545
tier3    0.553


**What surprised me:** team1 wins ~55% — not because of any game mechanic, but because
the source orders columns with the higher-seeded team first. It is harmless for Elo
(symmetric by construction) but a trap for any feature engineering that treats the
columns asymmetrically.
**What I'd do about it:** M7 trains on `diff` features only (already symmetric) and the
model report should note this bias explicitly.

## 3. Godsey-style decision paragraph

The dataset's genuine mess is concentrated in three places: the broken series
`team1_win` flag (never use it — see DATA.md Quirk 1), forfeit rows (0-0 or missing
map info — flag, don't drop, so the model layer can down-weight), and the two
generations of series rows (older ones carry negative game_ids). Everything else —
structural missingness on series rows, tier-file splits — is *by design* of the source
and costs nothing. The one decision with real model cost: dropping 2 corrupt rows
(9,923 -> 9,920) bought a trustworthy `winner` column at the price of 0.02% coverage —
a trade any reviewer should sign off on.